In [0]:
# Databricks notebook source

# COMMAND ----------
dbutils.widgets.removeAll()

dbutils.widgets.text("storage_account", "asaproyecto")
dbutils.widgets.text("storage_credential_name", "credential")

dbutils.widgets.text("raw_container", "raw")
dbutils.widgets.text("bronze_container", "bronze")
dbutils.widgets.text("silver_container", "silver")
dbutils.widgets.text("gold_container", "gold")

dbutils.widgets.text("external_location_raw", "exlt_raw")
dbutils.widgets.text("external_location_bronze", "exlt_bronze")
dbutils.widgets.text("external_location_silver", "exlt_silver")
dbutils.widgets.text("external_location_gold", "exlt_gold")

dbutils.widgets.text("read_principal", "datareaders")
dbutils.widgets.text("write_principal", "dataengineers")

# COMMAND ----------
storage_account = dbutils.widgets.get("storage_account")
storage_credential_name = dbutils.widgets.get("storage_credential_name")

raw_container = dbutils.widgets.get("raw_container")
bronze_container = dbutils.widgets.get("bronze_container")
silver_container = dbutils.widgets.get("silver_container")
gold_container = dbutils.widgets.get("gold_container")

external_location_raw = dbutils.widgets.get("external_location_raw")
external_location_bronze = dbutils.widgets.get("external_location_bronze")
external_location_silver = dbutils.widgets.get("external_location_silver")
external_location_gold = dbutils.widgets.get("external_location_gold")

read_principal = dbutils.widgets.get("read_principal")
write_principal = dbutils.widgets.get("write_principal")

locations = [
    {
        "name": external_location_raw,
        "url": f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/",
        "comment": "Ubicación externa para datasets raw"
    },
    {
        "name": external_location_bronze,
        "url": f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/",
        "comment": "Ubicación externa para tablas bronze"
    },
    {
        "name": external_location_silver,
        "url": f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/",
        "comment": "Ubicación externa para tablas silver"
    },
    {
        "name": external_location_gold,
        "url": f"abfss://{gold_container}@{storage_account}.dfs.core.windows.net/",
        "comment": "Ubicación externa para tablas gold"
    },
]

# COMMAND ----------
for loc in locations:
    stmt = f"""
    CREATE EXTERNAL LOCATION IF NOT EXISTS `{loc["name"]}`
    URL '{loc["url"]}'
    WITH (STORAGE CREDENTIAL `{storage_credential_name}`)
    COMMENT '{loc["comment"]}'
    """
    print("Ejecutando:")
    print(stmt.strip())
    spark.sql(stmt)

print("External locations creados correctamente.")

# COMMAND ----------
grant_statements = [
    f"GRANT READ FILES ON EXTERNAL LOCATION `{external_location_raw}` TO `{read_principal}`",
    f"GRANT READ FILES ON EXTERNAL LOCATION `{external_location_raw}` TO `{write_principal}`",
    f"GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION `{external_location_bronze}` TO `{write_principal}`",
    f"GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION `{external_location_silver}` TO `{write_principal}`",
    f"GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION `{external_location_gold}` TO `{write_principal}`",
]

for stmt in grant_statements:
    print("Ejecutando:")
    print(stmt)
    spark.sql(stmt)

print("Grants aplicados correctamente sobre external locations.")